## Imports

In [31]:
import serial
import time
import json
import numpy as np
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import PointCloud2
from sensor_msgs_py import point_cloud2
from IPython.display import clear_output
import altair as alt
import pandas as pd
from IPython.display import display
import ipywidgets as widgets
import Jetson.GPIO as GPIO
import requests
from pynput import mouse

## Robot Setup

You can either send commands to the robot using the serial port (through connection with the 40 pin uart, usb, etc) or through connecting with the wifi to the robot's hotspot. Adjust the useSerial variable to toggle.

In [32]:
useSerial = False

The pins for the ultrasonic sensors can be found in the GPIO setup.

### Communication Functions (Serial/Wifi)

It should be noted that the sendCmd function also returns the output for sensor related commands while using the wifi to communicate with the robot.

The serial sendCmd only writes the command to the serial port

In [33]:
if useSerial:
    def sendCmd(cmd):
        """Sends a command to the UGV through the serial port"""
        global ser
        
        # remove old data when sending new command
        ser.reset_input_buffer()
    
        jsonPayload = json.dumps(cmd) + "\n"
        ser.write(jsonPayload.encode('utf-8'))

    def openSerial():
        """Opens the serial port at /dev/ttyTHS1 with a baudrate of 115200"""
        global ser
        ser = serial.Serial(
            port = '/dev/ttyTHS1',
            baudrate = 115200,
            bytesize = serial.EIGHTBITS,
            parity = serial.PARITY_NONE,
            stopbits = serial.STOPBITS_ONE,
            timeout = 2.0           
        )
        sendCmd({"T":900,"main":2,"module":2})
        sendCmd({"T":2,"P":200,"I":2500,"D":0,"L":255})

    def closeSerial():
        """Close the serial port"""
        global ser
        ser.close()
else:
    # this is the url that the jetson nano communicates with the ugv through
    ugvURL = "http://192.168.4.1/js"

    def openSerial():
        """Does nothing"""
        pass
    def closeSerial():
        """Does nothing"""
        pass

    def sendCmd(cmd):
        """Sends a command through the ugv's web based interface"""
        global ugvURL
        response = requests.post(ugvURL, json = cmd)
        if response.text:
            return response.json()
        return response.text

### Sensor Functions (Serial/Wifi)

In [34]:
def readSensor(sensor):
    """Read the data for a specified sensor, returning a 2d list with labels in the first row and data in the second"""
    if sensor == "imu":
        cmd = {"T": 126}
    elif sensor == "motor":
        cmd = {"T": 130}

    # if using the serial port, send the command and then read lines from the serial port until the data is found
    if useSerial:                   
        global ser
        sendCmd(cmd)
        
        # remove lines until the line that contains the command to get the data is found
        foundLine = False
        while not foundLine:
            line = ser.readline().decode('utf-8').strip()
            linesplit = str(line).replace(" ", "").replace("{", "").replace("}", "").split(":")
            cmdsplit = str(cmd).replace(" ", "").replace("{", "").replace("}", "").split(":")
            if linesplit[1] == cmdsplit[1]:
                foundLine = True
                
        # read the line with the data in it
        line = ser.readline().decode('utf-8').strip()
    # otherwise send the command to the url and directly recieve the response
    else:
        line = sendCmd(cmd)

    # clean data
    line = line.replace("\"", "").replace("{", "").replace("}", "")
    dataFull = line.split(",")
    
    # create array with 2 rows, first row is the name of the value and the second row is the value
    data = []
    data.append([])
    data.append([])
    for value in dataFull:
        valueList = value.split(":")
        data[0].append(valueList[0])
        data[1].append(valueList[1])
    return data

In [35]:
def getValFromData(value, data = [], sensor = ""):
    """Looks for a specific label in the first row of a 2d array of data and returns the corresponding values found in the second row"""
    if not sensor == "":
        data = readSensor(sensor)
    position = data[0].index(value)
    val = data[1][position]
    return val

In [36]:
# get data for the motor, specifically how much the left and right motors have moved (over all movement)
totalLValue = totalRValue = 0.0
lValue = rValue = 0.0

def updateMotorData():
    """Updates the variables that represent the values for the left and right motor encoders. Should be run after the robot is moved."""
    global lValue, rValue, totalLValue, totalRValue
    
    lValue = round(float(getValFromData("L", sensor = "motor")), 3)
    rValue = round(float(getValFromData("R", sensor = "motor")), 3)
    totalLValue += lValue
    totalRValue += rValue
    totalLValue = round(totalLValue, 3)
    totalRValue = round(totalRValue, 3)

def getMotorData(printData = False):
    """Returns the values of the left and right motor encoders. Setting printData to true prints these values as well as how much they changed after the most recent update of their values."""
    global lValue, rValue, totalLValue, totalRValue
    
    recentTotal = [lValue, rValue]
    total = [totalLValue, totalRValue]
    if printData:
        print(f"retrieved motor data: {total}\nnew: {recentTotal}")
    return total

def resetMotorData():
    """Reset all motor data values to 0."""
    global lValue, rValue, totalLValue, totalRValue
    lValue = rValue = totalLValue = totalRValue = 0
    
def checkIfStraight():
    """Check to see if one of the motor encoders has a higher value."""
    motorData = getMotorData()
    if abs(motorData[0] - motorData[1]) > 0.05:
        return False
    else: 
        return True

### GPIO Setup

In [37]:
GPIO.setmode(GPIO.BOARD)

trigPins = [29, 33]
echoPins = [31, 32]

# ultrasonic sensor 1
GPIO.setup(trigPins[0], GPIO.OUT)
GPIO.setup(echoPins[0], GPIO.IN)

# ultrasonic sensor 2
GPIO.setup(trigPins[1], GPIO.OUT)
GPIO.setup(echoPins[1], GPIO.IN)

### Sensor Functions (GPIO)

In [38]:
def getUltrasonicDistance(side):
    """Get the distance values (meters) of one of the two ultrasonic sensors."""
    if side == "left":
        trigPin = trigPins[0]
        echoPin = echoPins[0]
    else:
        trigPin = trigPins[1]
        echoPin = echoPins[1]

    # get pins and stuff ready
    GPIO.output(trigPin, GPIO.LOW)
    time.sleep(0.1)
    
    GPIO.output(trigPin, GPIO.HIGH)
    time.sleep(0.00001)
    GPIO.output(trigPin, GPIO.LOW)
    
    startTime = stopTime = 0
    timeoutTime = time.time()
    
    # wait for echo pin to go high
    while GPIO.input(echoPin) == 0:
        startTime = time.time()
        if time.time() - timeoutTime > 0.1:
            return -1
            
    # Wait for echo pin to go low
    while GPIO.input(echoPin) == 1:
        stopTime = time.time()
        if time.time() - timeoutTime > 0.1:
            return -1

    # calculate distance (m) based on time between pin going high/low
    elapsed_time = stopTime - startTime
    distance = round((elapsed_time * 34300) / 200, 4)
    return distance

In [39]:
def getDistFull():
    """Get the combined distance of the two ultrasonic sensors."""
    distLeft = getUltrasonicDistance("left")
    distRight = getUltrasonicDistance("right")
    distFull = distLeft + distRight
    return distFull

def getDistBothSides():
    """Get the values of the two ultrasonic sensors individually."""
    distLeft = getUltrasonicDistance("left")
    distRight = getUltrasonicDistance("right")
    return distLeft, distRight

In [40]:
def getDistStableFull():
    """Get the average of 10 values for the combined distance of the two ultrasonic sensors"""
    distFullAvg = 0
    for distances in range(10):
        distFullAvg += getDistFull()
    distFullAvg /= 10
    return distFullAvg

def getDistStableBothSides():
    """Get the average of 10 values for the individual distances of the two ultrasonic sensors"""
    distLeftAvg = distRightAvg = 0
    for distances in range(10):
        distLeftNew, distRightNew = getDistBothSides
        distLeftAvg += distLeftNew
        distRightAvg += distRightNew
    distLeftAvg /= 10
    distRightAvg /= 10
    return distLeftAvg, distRightAvg

### Movement Functions

In [41]:
def stopMoving():
    """Stop moving the robot and update the motor data."""
    updateMotorData()
    motionCmd = {"T":1,"L":0,"R":0}
    sendCmd(motionCmd)

In [42]:
def moveForward(vLeft, vRight = -1, moveTime = -1):
    """Move forward for a set amount of time. If vRight is not set, assume it is the same as vLeft. Velocity is in meters per second."""
    if vRight == -1:
        vRight = vLeft
    motionCmd = {"T":1,"L":vLeft,"R":vRight}
    sendCmd(motionCmd)

    if moveTime != -1:
        time.sleep(moveTime)
        stopMoving()
        time.sleep(0.2)

def moveBackward(vLeft, vRight = -1, moveTime = -1):
    """Move backward for a set amount of time. If vRight is not set, assume it is the same as vLeft. Velocity is in meters per second."""
    if vRight == -1:
        vRight = vLeft
    moveForward(-vLeft, -vRight)

    if moveTime != -1:
        time.sleep(moveTime)
        stopMoving()
        time.sleep(0.2)

In [43]:
def turn(speed, angle, moveTime = 1):
    """Turn the robot at a specified angle (degrees)."""
    angleRadians = angle / 57.2958
    motionCmd = {"T":13,"X":speed,"Z":angleRadians}
    sendCmd(motionCmd)

    time.sleep(moveTime)
    stopMoving()
    time.sleep(0.2)

In [44]:
def straighten(printData = False):
    """If one of the motor encoders has a higher value attempt to equalize."""
    while not checkIfStraight():
        if printData:
            clear_output(wait = True)
        motorData = getMotorData(printData)
        if motorData[0] > motorData[1]:
            turn(0.05, -5, 0.5)
        elif motorData[1] > motorData[0]:
            turn(0.05, 5, 0.5)
        else:
            break

### Testing Space

## Lidar Setup

Robot navigates by splitting the path ahead of it into sections, each slightly larger than the robot (25 cm, or double the ySensitivity variable). Therefore, during navigation, the robot will only see about 0.25 * the number of sections meters.

The robot in cave mode will navigate assuming that the first and last sections will be wall if the robot is in the center, meaning that for a tunnel of 1m radius there should be at least 11 sections, 2m 22 sections, etc. Default is 7 sections which assumes a tunnel of a diameter a bit less than a meter. Having extra sections makes for a more robust navigation program but can result in a heavier load on the host computer.

In [45]:
# must be odd
numberOfSections = 7

Note that if the zSensitivity if off, the robot may see the floor or ceiling of the tunnel and think it is an obstacle.

### Node Setup

This node setup is based on the Livox Avia lidar, utilizing ASIG-X's driver (https://github.com/ASIG-X/livox_ros2_avia) to publish data. May need adjustment if the lidar is changed.

Also the Avia has a large 1 meter blind spot and is not a 360 degree lidar which makes navigation much more difficult. Adjustments could be made if using a more suited lidar that would result in better/more efficient navigation.

In [46]:
# initialize rclpy
rclpy.init()

In [47]:
# create a node to recieve and array to store pointcloud data
nodeLidar = Node('lidarNode')
pcDataList = []
for sectionNum in range(numberOfSections):
    pcDataList.append(np.empty((0, 3)))

# determines how close the points scanned have to be in terms of width/height to the lidar (cm)
ySensitivity = 12.5
zSensitivity = 6

In [48]:
def updateLidarData(msg):
    """Takes in new data 'msg' and updates pcDataList based on the input."""
    global pcDataList
    global ySensitivity, zSensitivity
    global numberOfSections
    
    points = []
    for sectionNum in range(numberOfSections):
        for point in point_cloud2.read_points(msg, field_names = ('x','y','z'), skip_nans = True):
            # distance of point from robot
            distance = np.sqrt(point[0]**2 + point[1]**2 + point[2]**2)

            # don't accept points that are too close or points that are too far away from lidar vertically (floor + ceiling)
            if distance > 0.1 and (zSensitivity / -100) < point[2] < (zSensitivity / 100):
                # determines how far to the left of right data section will be
                # 3 sections to the left, 3 to the right, and one in the center are created respective of the robot (each section is slightly larger than the robot in width)
                if (ySensitivity / -100) * (-3 + sectionNum) < point[1] < (ySensitivity / 100) * (3 + sectionNum):
                    points.append([point[0], point[1], point[2]])
        
        # check if any data is logged, as logging blanks can change shape of array
        if len(points) == 0:
            pcDataList[sectionNum] = np.empty((0, 3))
            print(f"added 0 points to section {sectionNum}")
        else:
            # if there is data, add it to the corresponding section
            pcDataList[sectionNum] = np.array(points)
            print(f"added {len(points)} points to section {sectionNum}")

In [49]:
# subscribe to /livox/lidar node to recieve point cloud data and update data
node.create_subscription(PointCloud2, '/livox/lidar', updateLidarData, 10)

In [ ]:
# short test to see if the sections are properly adding data.
rclpy.spin_once(nodeLidar, timeout_sec=2)
print(len(pcDataList))
print(len(pcData))

### Lidar Functions

In [ ]:
def getLMiddleIndex():
    """Return the index of the section in the middle (directly in front of the robot)"""
    global numberOfSections
    index = (numberOfSections - 1)/2
    return index

In [ ]:
def getLSectionNum():
    """Get the total number of sections"""
    global numberOfSections
    return numberOfSections

In [ ]:
def getClosestPointInfo(data):
    """Gets data about the point closest to lidar in a given data set."""
    # get the distances of each point
    distances = np.sqrt(data[0:,0]**2 + data[:,1]**2)
    
    # find the index of the point with the smallest distance
    closestPointIndex = np.argmin(distances)
    
    # find the point based on the index and get its data
    closestPoint = data[closestPointIndex]
    pointInfo = {
        "distance": distances[closestPointIndex],
        "x": closestPoint[0],
        "y": closestPoint[1],
        "z": closestPoint[2],
        "closestPoint": closestPoint
    }
    return pointInfo

In [ ]:
def getPointCloudData(clear = False, cont = False, update = True, graph = False, iteration = 0, erriteration = 0, iterationMax = -1):
    """Testing function that shows information about the section directly in front of the robot"""
    global pcDataList
    global jchart
    global textDisplay
    global ySensitivity

    pcData = pcDataList[getLMiddleIndex()]
    
    try:
        # if update is true update to get fresh data
        if update:
            rclpy.spin_once(nodeLidar, timeout_sec=2)
            
        pointNum = pcData.shape[0]
    
        # if clear is true clear the current terminal (automatically on if cont is true to prevent clutter)
        if clear or (cont and not graph):
            clear_output(wait = True)
    
        # checks to see if there are valid points to look at 
        if pointNum > 0:
            # retrieves and displays data
            closestPointInfo = getClosestPointInfo(pcData)
            t1 = (f"recieved {pointNum} points<br>closest point info:<ul><li>coordinates: {closestPointInfo.get('closestPoint')}</li><li>dist: {closestPointInfo.get('distance')}</li><li>dist (forward only): {round(np.sqrt(closestPointInfo.get('x')**2),2)}</li></ul>")
            if cont:
                t2 = (f"iteration num: {iteration}")
                if iteration == 0:
                    textDisplay = widgets.HTML(value = f"{t1}{t2}")
                    display(textDisplay)
                else:
                    textDisplay.value = f"{t1}{t2}"
            else:
                print(t1)
        else:
            t1 = ("recieved 0 points")
            if cont:
                t2 = (f"iteration num: {iteration}")
                if iteration == 0:
                    textDisplay = widgets.HTML(value = f"{t1}<br>{t2}")
                    display(textDisplay)
                else:
                    textDisplay.value = f"{t1}<br>{t2}"
            else:
                print(t1)

        # create a graph to visualize lidar readings if graph is true
        if graph:
            # set x data to first column and y data to second column of pcData
            gdata = pd.DataFrame(pcData[:, :2], columns = ["x", "y"])

            # make chart of robot view
            chart = alt.Chart(gdata).mark_circle(size = 50).encode(
                x = alt.X('x:Q', scale = alt.Scale(domain = [0, 8])),
                y = alt.Y('y:Q', scale = alt.Scale(domain = [(ySensitivity / -100), (ySensitivity / 100)]))
            ).properties(
                title = "robot view"
            )

            # use jupyter charts to update graph instead of making a new one each time
            if iteration == 0:
                # define jcharts and initially display
                jchart = alt.JupyterChart(chart)
                display(jchart)
            else:
                # update jcharts and then update display
                jchart.chart = chart
                jchart
        
        # runs again if cont is true
        if cont:
            iteration += 1
            if iterationMax == -1 or iteration < iterationMax:
                getPointCloudData(cont = True, graph = graph, iteration = iteration)
    except Exception as e:
        # output information about error encountered to help with troubleshooting
        erriteration += 1
        clear_output(wait = True)
        print(f"exited with exception \"{e}\"\ncompilation failed {erriteration} times")

        # if this error occurs it's probably this so just added a warning
        if isinstance(e, ValueError):
            print("\nlidar is likely being blocked by something")
            
        # iterations must be reset to 0 when returning from exception code to regular code so graph is reset
        getPointCloudData(cont = True, clear = True, graph = graph, erriteration = erriteration)

### Testing Space

## Navigation

### Navigation Variables

In [52]:
# initialize variables
navigating = navigatingForward = True
navigatingBackward = False
relativeCenter = 0
halfDistance = 999

inCave = False
testing = True
mouseless = False

blockedSectionList = []
blockedSectionList.append([])
blockedSectionList.append([])
blockedSectionList.append([])
sectionPositions = []

In [ ]:
# distance at which an obstacle is recognized
blockedDist = 1.3
# distance at which a dead end is recognized
blockedDistMax = 1.5
# weight added to paths that are closer to the center of the tunnel
centerWeight = 1.1
# weight added to sections closer to the opposite wall if robot cannot see wall
centerWeightNoWall = 1.3
# weight added to paths that are closer to the current path
stayWeight = 10
# weight reduced from paths that are blocked
blockedWeight = 10

### Navigation Functions

In [ ]:
def checkIfRobotSeesTwoWalls():
    """Checks the leftmost section and the rightmost section to determine if the robot can see the walls."""
    global pcDataList
    
    if len(pcDataList[0]) == 0 and len(pcDataList[getLSectionNum() - 1] == 0):
        return True
    else:
        return False

In [ ]:
def updateNavData():
    """Get fresh data from the lidar and navigation data based on that."""
    global pcDataList
    
    # update data
    rclpy.spin_once(nodeLidar, timeout_sec = 2)
    
    distanceList = []
    distanceListWeights = []
    for sectionNum in range(getLSectionNum()):
        try:
            # get the closest point for each section (locates how far the robot can go in each section since each section is about the size of the robot)
            sectionInfo = getClosestPointInfo(pcDataList[sectionNum])
            
            # add the points to a list to store all the distances
            distanceList.append(sectionInfo.get('distance'))
            
            # weighted version that favors sections closer to the robot
            distanceListWeights.append(sectionInfo.get('distance') * (1 - abs(sectionNum - 3) / stayWeight))
        except:
            # if previous action fails the section likely has 0 points in it and is being blocked by something so just append 0
            distanceList.append(0)
            distanceListWeights.append(0)

    return distanceList, distanceListWeights

In [ ]:
def updateBlockedSections(distanceList, closestObstacleSectionIndex, furthestObstacleSectionIndex, progress):
    """Update the list of blocked sections based on fresh data. Remove old blocked sections."""
    global blockedSectionList, relativeCenter
    
    # if there are blocked sections, add all them to the list
    if distanceList[closestObstacleSectionIndex] < blockedDist:
        for sectionNum in range(getLSectionNum()):
            if distanceList[sectionNum] < blockedDist:
                blockedSectionList[0].append(sectionNum)
                blockedSectionList[1].append(progress)
                blockedSectionList[2].append(relativeCenter)
    
    # mark old sections from blocked section list
    indexesToRemove = []
    for index, blockedSectionDist in enumerate(blockedSectionList[1]):
        if progress[0] - blockedSectionDist > blockedDist:
            indexesToRemove.append(index)
    
    # remove old sections starting from last to prevent errors
    for index in reversed(indexesToRemove):
        blockedSectionList[0].pop(index)
        blockedSectionList[1].pop(index)
        blockedSectionList[2].pop(index)

In [ ]:
def updateNavWeights(distanceListWeights):
    """Update the weights of each section based on fresh data."""
    global blockedSectionList, inCave
    listIndex = 0
    
    # make it so robot tries not to go across or to blocked sections to get to goal
    # (does not completely eliminate in case only way through is through a blocked section)
    for blockedSection in blockedSectionList[0]:
        relativeBlockedSection = blockedSection + blockedSectionList[2][listIndex]
        listIndex += 1

        if 0 <= relativeBlockedSection <= getLSectionNum() - 1:
            if relativeBlockedSection < getLMiddleIndex():
                for sectionsBlockedLeft in range(blockedSection + 1):
                    distanceListWeights[sectionsBlockedLeft] /= blockedWeight
            elif relativeBlockedSection > getLMiddleIndex():
                for sectionsBlockedRight in range(blockedSection - getLMiddleIndex()):
                    distanceListWeights[sectionsBlockedRight + getLMiddleIndex()] /= blockedWeight
            else:
                distanceListWeights[relativeBlockedSection] /= blockedWeight
    
    # weigh sections that bring the robot closer to the center higher
    if relativeCenter != 0:
        if relativeCenter < 0:
            for sectionsLeft in range(getLMiddleIndex()):
                distanceListWeights[sectionsLeft] *= centerWeight
        elif relativeCenter > 0:
            for sectionsRight in range(getLMiddleIndex()):
                distanceListWeights[sectionsRight + getLMiddleIndex() + 1] *= centerWeight
        
    
    # weigh sections that bring the robot further from the wall it can see if it can't see the other wall
    if inCave:
        if len(pcDataList[0]) > 0:
            for sectionsLeft in range(getLMiddleIndex()):
                distanceListWeights[sectionsLeft] *= centerWeightNoWall
        elif len(pcDataList[getLSectionNum()]) > 0:
            for sectionsRight in range(getLMiddleIndex()):
                distanceListWeights[sectionsRight + getLMiddleIndex() + 1] *= centerWeightNoWall

    return distanceListWeights

### Navigation Functions (Movement Related)

In [ ]:
def moveToSection(section):
    """Move to a specific section by turning 90 degrees, moving a certain amount, then turning back."""
    sectionsToMove = getLMiddleIndex() - section
    if sectionsToMove < 0:
        sectionsToMove = abs(sectionsToMove)
        turn(0.1, 90)
        moveForward(ySensitivity / 100, moveTime = 2 * sectionsToMove)
        turn(0.1, -90)
    elif sectionsToMove > 0:
        turn(0.1, -90)
        moveForward(ySensitivity / 100, moveTime = 2 * sectionsToMove)
        turn(0.1, 90)

In [50]:
def onClick(x, y, button, pressed):
    # check if mouse button was pressed down
    if pressed:
        return False

    
def waitForInput():
    """Wait for a specific input continue the program. If there is no mouse, wait for a command to be sent through wifi by an external device. Otherwise, wait for a wireless mouse to be clicked. Since using the wifi mode requires the ugv hotspot to be connected to the nano, a mouse should be used unless using the serial mode."""
    global mouseless
    if mouseless:   
        waiting = True
        motorDataOld = getMotorData()

        while waiting:
            try:
                # get the motor data and see if wheels have moved
                updateMotorData()
                motorData = getMotorData()
                if motorData[0] == motorDataOld[0] and motorData[1] == motorDataOld[0]:
                    print("waiting")
                else:
                    waiting = False
            except:
                print("waiting")
            clear_output(wait = True)
            time.sleep(0.2)
    else:
        # wait until a mouse button is clicked
        with mouse.Listener(on_click=onClick) as listener:
            listener.join()

In [ ]:
def moveStraight(segmentLength):
    """Move forward 10 times, moving the segment length (meters) each time. Attempt to prevent drifting through several measures while moving."""
    # attempt to equalize left and right motor encoders before moving
    straighten()
        
    distLeft = distRight = distLeft2 = distRight2 = differenceAvg = 0
    for segments in range(10):
        # find if it is moving closer or further to the left/right wall as it goes
        differenceLeft = distLeft - distLeft2
        differenceRight = distRight - distRight2
        differenceAvg = (abs(differenceLeft) + abs(differenceRight)) / 2
                
        distLeft, distRight = getDistStableBothSides()
        # if it is tilted a specific amount begin correcting
        if differenceAvg > 0.1:
            # if differenceLeft is positive, it is getting closer to left and needs to move more right and vice versa with differenceRight
            # moving higher speed for vLeft or vRight will make the robot move more towards the right or left respectively, so adding the difference to the velocity should be effective in correcting
            moveForward(0.1 + differenceLeft, vRight = 0.1 + differenceRight, moveTime = (segmentLength))
        else:
            moveForward(0.1, moveTime = (segmentLength))
        distLeft2, distRight2 = getDistStableBothSides()

### Navigation Loop

When starting the code the robot will wait for an input (typically a mouse press), and then upon recieiving the input it will run the navigation loop. 

The robot will navigate forward until running into a dead end, then it will turn around and navigate back until reaching its starting position.

In [54]:
# wait for robot to be sent move forward cmd to begin
openSerial()
resetMotorData()
waitForInput()
closeSerial()
print("successfully recieved signal, beginning navigation")

successfully recieved signal, beginning navigation


In [ ]:
print(len(pcDataList))

openSerial()
resetMotorData()

while navigating:
    clear_output(wait = True)
    print("navigation cycle beginning\n")
    try:
        # if robot has returned to the starting position signal that navigation has finished
        progress = getMotorData()
        if (progress[0] > halfDistance) and not navigatingForward:
           break
        
        distanceList, distanceListWeights = updateNavData()

        closestObstacleSectionIndex = distanceList.index(min(distanceList))
        furthestObstacleSectionIndex = distanceList.index(max(distanceList))

        updateBlockedSections(distanceList, closestObstacleSectionIndex, furthestObstacleSectionIndex, progress)

        distanceListWeights = updateNavWeights(distanceListWeights)

        optimalSectionToGoToIndex = distanceListWeights.index(max(distanceListWeights))

        if testing:
            print(f"\nclosest obstacle at section {closestObstacleSectionIndex + 1}, {distanceList[closestObstacleSectionIndex]} m away")
            print(f"furthest obstacle at section {furthestObstacleSectionIndex + 1}, {distanceList[furthestObstacleSectionIndex]} m away\n")
            print(f"section distances: {distanceList}")
            print(f"section weights: {distanceListWeights}\n")
            print(f"optimal section to move to is {optimalSectionToGoToIndex + 1} with distance of {distanceList[optimalSectionToGoToIndex]} and weight of {distanceListWeights[optimalSectionToGoToIndex]}")
            if optimalSectionToGoToIndex < getLMiddleIndex():
                print("move left")
            elif optimalSectionToGoToIndex > getLMiddleIndex():
                print("move right")
            else:
                print("stay on current path")
        
        # if obstacle is directly ahead or robot is not in original center begin thinking about moving to an alternate section to avoid
        if distanceList[getLMiddleIndex()] < blockedDist or relativeCenter != 0:
            # checks to see if all paths ahead are blocked if they are blocked, a dead end is likely ahead
            if distanceList[furthestObstacleSectionIndex] < blockedDistMax:
                # if a path is clear until past the robot can see 0 points will be logged in all sections which could result in false block
                distanceTotal = 0
                for distance in distanceList:
                    distanceTotal += distance

                # check if false block or actually blocked
                if distanceTotal == 0:
                    moveForward(0.1, moveTime = 10)
                else:
                    print(f"furthest obstacle only {distanceList[furthestObstacleSectionIndex]} m away, likely reached dead end")
                    
                    if inCave:
                        # center robot if not centered so that it can see the entire tunnel, as it could be very far to the left or right preventing it from seeing a path
                        if not checkIfRobotSeesTwoWalls():
                            if len(pcDataList[0]) > 0:
                                print("must move left")
                                moveToSection(getLMiddleIndex() - 1)
                                relativeCenter -= 1
                            elif len(pcDataList[getLSectionNum()]) > 0:
                                print("must move right")
                                moveToSection(getLMiddleIndex() + 1)
                                relativeCenter += 1
                        # if the robot is centered then it is very likely a dead end
                        else:
                            # if going forward begin returning
                            if navigatingForward:
                                navigatingForward = False
                                navigatingBackward = True
                                turn(0.1, 90, moveTime = 2)
                                halfDistance = progress[0]
                                resetMotorData()
                            # if it isn't going forward this means the robot got stuck coming back
                            else:
                                print("robot is cooked")
                                break
                    # if not in cave just exit
                    else:
                        break
            else:
                print(f"object ahead approaching recommended to move to section {optimalSectionToGoToIndex + 1}")
                moveToSection(optimalSectionToGoToIndex)
                relativeCenter += optimalSectionToGoToIndex - getLSectionNum()
                moveStraight(distanceList[optimalSectionToGoToIndex] - blockedDist * 1.01)
        # no obstacle ahead just keep going
        else:
            moveStraight(distanceList[getLMiddleIndex()] - blockedDist * 1.01)
        
        if testing:
            print("send cmd to robot to continue")
            waitForInput()
    # in an exception loop because if something is blocking the lidar it runs into an error
    except Exception as e:
        print(f"failed to navigate, ran into error {e}")
        
        if isinstance(e, ValueError):
            print("\nlidar is likely being blocked by something")

        if isinstance(e, KeyboardInterrupt):
            print("keyboard interrupt")
            break

closeSerial()
print("navigation finished")